# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. You will learn to inspect metadata, list available record sets and fields, and perform basic data analysis and visualization, all while referencing record sets, fields, and columns by their `@id` fields for reproducibility.

### Dataset Source
The Croissant schema describing this dataset is available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
We'll start by loading metadata and discovering the available record sets using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
pd.set_option('display.max_columns', None)

# Define the Croissant schema URL (FAIR² dataset)
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load metadata and dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}\n")

## 2. Data Overview
Next, we examine which record sets, fields, and columns are available, referencing their unique `@id` fields as required by the Croissant specification.

We'll print a list of record set `@id`s and, for each, list the fields and their types.

In [ ]:
# List all record sets by their @id fields
print('Record sets in this dataset:')
record_set_objs = dataset.metadata.record_sets
if not record_set_objs:
    print('No record sets found in metadata.')
else:
    for rs in record_set_objs:
        print(f"- RecordSet @id: {rs['@id']}")
        print("  Name:", rs.get('name', '[No name]'))
        if 'fields' in rs:
            print("  Fields:")
            for fld in rs['fields']:
                fid = fld.get('@id', '[No @id]')
                fname = fld.get('name', '[No name]')
                ftype = fld.get('dataType', '[No dataType]')
                print(f"    - Field @id: {fid} | Name: {fname} | Type: {ftype}")
        print()

Let's fetch the complete list of record sets programmatically for further analysis and load their records using their `@id` fields. To continue, we'll select the primary clinical data record set for demonstration.

*Note: If you are unsure which record set to use, refer to the previous output for available options*.

In [ ]:
# Get all record_set @id values
record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]
print('All record_set @id values:', record_sets)

## 3. Data Extraction
We now load data from a specific record set into a DataFrame for analysis. All references use the record set and field `@id` identifiers.

For the FAIR² dataset, we use the primary tabular record set. Based on the metadata, let's extract the main clinical record set (assuming only one record set is present).

In [ ]:
# For most clinical tabular datasets, there is only one record set.
main_record_set_id = record_sets[0] if len(record_sets) else None
print('Selected record_set @id for DataFrame:', main_record_set_id)

dataframes = {}

if main_record_set_id is not None:
    # Load all records using the record_set @id
    records = list(dataset.records(record_set=main_record_set_id))
    df = pd.DataFrame(records)
    dataframes[main_record_set_id] = df
    print(f"Loaded {len(df)} records. Columns (field @id):")
    print(df.columns.tolist())
    display(df.head())
else:
    print('No record sets available to extract.')

## 4. Exploratory Data Analysis (EDA)
We now process some fields using their `@id`. We'll select a numerical field (such as age), filter for records exceeding a threshold, normalize the values, and optionally group by a categorical field (such as sex or MSI status). All variable references use field `@id`s.

In [ ]:
# Inspect the columns to determine numeric/categorical fields
numeric_field = None
group_field = None

if main_record_set_id:
    df = dataframes[main_record_set_id]
    print('Available columns (field @id):', df.columns.tolist())
    # Optionally, print first row for more context
    print('\nSample row:')
    print(df.iloc[0])

    # Try to auto-select fields if typical field names are found
    # We'll prefer @id fields containing 'age' for numeric, 'sex' or 'msi' for group
    for c in df.columns:
        if 'age' in c.lower() and numeric_field is None:
            numeric_field = c
        if ('sex' in c.lower() or 'msi' in c.lower() or 'status' in c.lower()) and group_field is None:
            group_field = c

    # Confirm selected fields
    print(f'Numeric field selected (for analysis): {numeric_field}')
    print(f'Group field selected (for grouping): {group_field}')

    # Convert numeric field to numeric if not already
    if numeric_field:
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

        threshold = df[numeric_field].mean() if not pd.isnull(df[numeric_field].mean()) else 10

        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"\nFiltered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        mean = filtered_df[numeric_field].mean()
        std = filtered_df[numeric_field].std()
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std

        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # If there is a group field, group by its values
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame('mean_' + numeric_field)
            print(f"\nGrouped mean {numeric_field} by {group_field}:")
            display(grouped_df)
else:
    print('No record set loaded for EDA analysis.')

## 5. Visualization
We can now visualize the distribution of a numeric field (such as age) and analyze relationships (such as by group field).

All plots reference the correct field `@id`.

**Note:** If no numeric or grouping field was found, update `numeric_field` and/or `group_field` variables above accordingly.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field:
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=15, color='dodgerblue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If group field exists, compare distributions
    if group_field and group_field in df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print('No numeric/group field found for plotting.')

## 6. Conclusion
In this notebook, we demonstrated how to load and explore clinicopathological and molecular data from the FAIR² dataset using the `mlcroissant` library.

- We loaded metadata and identified record sets by their `@id`.
- We extracted the main record set and accessed all field columns by their `@id` fields.
- We performed basic EDA, filtering and normalizing a numeric column, and optionally grouped by a categorical field.
- Finally, we visualized data distributions and group comparisons.

Be sure to always use `@id` for referencing entities and fields, which ensures reproducibility and accurate documentation when working with Croissant-conformant datasets.